# ArXiv RAG — Stage 2: Embedding Pipeline

Reads the papers already in Neon Postgres (from Stage 1), chunks each abstract,
embeds each chunk with `nomic-embed-text-v2-moe` on GPU, and bulk-inserts into
the `chunks` table (`content`, `embedding`). No LLM calls are made in this
pipeline — contextual blurb generation (Anthropic's Contextual Retrieval) was
dropped from the project to avoid burning through free-tier LLM rate limits;
hybrid vector+BM25 retrieval works fine without it.

**Works on both Kaggle and Google Colab** — the secrets-loading cell below detects which platform you're on.

## Before running — Colab setup

1. **Runtime → Change runtime type → T4 GPU** (top menu)
2. Click the **key icon 🔑** in the left sidebar (Secrets panel)
3. Add this secret (toggle "Notebook access" on):
   - `DATABASE_URL` — `postgresql://neondb_owner:<password>@<host>/neondb?sslmode=require&channel_binding=require`
4. **Runtime → Run all**

## Before running — Kaggle setup (unchanged)

1. **Add-ons → Secrets** — add `DATABASE_URL`
2. **Settings → Accelerator → GPU T4**
3. **Run All**

## Note on re-running

This pipeline inserts chunks for whatever papers are fetched from `papers` —
it does not skip or dedupe already-embedded papers. If you re-run it against
the same `limit`, you will get duplicate chunk rows for previously-processed
papers. Adjust the `LIMIT`/query in the pipeline cell if you need incremental
runs over a large corpus.

In [ ]:
# Install dependencies
!pip install -q 'psycopg[binary]' psycopg_pool 'pgvector>=0.3.5' sentence-transformers einops numpy openai

In [ ]:
import os

def _load_secret(name: str) -> str | None:
    """Load a secret from Colab userdata, Kaggle secrets, or existing env var."""
    # Try Colab
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # Try Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

for _key in ("DATABASE_URL",):
    _val = _load_secret(_key)
    if _val:
        os.environ[_key] = _val
        print(f"{_key} loaded")
    else:
        print(f"WARNING: {_key} not found — set it manually: os.environ['{_key}'] = '...'")

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import logging
import time
import re
from dataclasses import dataclass

from psycopg_pool import AsyncConnectionPool
from pgvector.psycopg import register_vector_async
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Config ──────────────────────────────────────────────────────────────────
EMBED_MODEL   = 'nomic-ai/nomic-embed-text-v2-moe'
EMBED_DIM     = 768
CHUNK_SIZE    = 400   # tokens (approx words)
CHUNK_OVERLAP = 50
BATCH_SIZE    = 1000  # papers per flush
EMBED_BATCH   = 512   # texts per model.encode() call
LIMIT         = 50_000

print('Config loaded')

In [ ]:
# ── Chunking ─────────────────────────────────────────────────────────────────
@dataclass
class Chunk:
    doc_id: str
    section_title: str
    chunk_index: int
    content: str
    token_count: int

def _word_chunks(text: str, size: int, overlap: int) -> list[str]:
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start += size - overlap
    return chunks

def chunk_text(text: str, doc_id: str, section_title: str = 'abstract') -> list[Chunk]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    raw = _word_chunks(text, CHUNK_SIZE, CHUNK_OVERLAP)
    return [
        Chunk(doc_id=doc_id, section_title=section_title,
              chunk_index=i, content=c, token_count=len(c.split()))
        for i, c in enumerate(raw)
    ]

print('Chunking helpers ready')

In [ ]:
# ── Embedding model ───────────────────────────────────────────────────────────
print(f'Loading {EMBED_MODEL} ...')
_model = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=device)
print(f'Model loaded (dim={EMBED_DIM})')

def embed_chunks(chunks: list[Chunk]) -> list[tuple[Chunk, list[float]]]:
    if not chunks:
        return []
    texts = [f'search_document: {c.content}' for c in chunks]
    embeddings = _model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=EMBED_BATCH,
        device=device,
    )
    return list(zip(chunks, embeddings.tolist()))

# Quick smoke test
test = embed_chunks([Chunk('test', 'abstract', 0, 'attention is all you need', 5)])
assert len(test[0][1]) == EMBED_DIM
print(f'Smoke test passed — embedding dim={len(test[0][1])}')

In [ ]:
# ── Database ──────────────────────────────────────────────────────────────────
_pool = None

async def init_pool():
    global _pool
    if _pool is not None:
        await _pool.close()
    url = os.environ['DATABASE_URL']
    _pool = AsyncConnectionPool(conninfo=url, min_size=1, max_size=5, open=False)
    await _pool.open()
    logger.info('DB pool initialized')

async def close_pool():
    global _pool
    if _pool:
        await _pool.close()

async def insert_chunks_batch(conn, chunks: list[dict]):
    await register_vector_async(conn)
    sql = """
        INSERT INTO chunks (paper_id, section_title, chunk_index, content, token_count, embedding)
        VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = [
        (c['paper_id'], c['section_title'], c['chunk_index'],
         c['content'], c['token_count'], c['embedding'])
        for c in chunks
    ]
    async with conn.cursor() as cur:
        await cur.executemany(sql, params)

async def write_batch_with_retry(rows: list[dict], max_attempts: int = 2):
    """Insert one batch, committing once. Neon's pooler can drop idle connections,
    so on failure we reinitialize the pool and retry once (a plain loop, not a
    re-entered context manager — asynccontextmanager only yields once)."""
    last_exc = None
    for attempt in range(max_attempts):
        try:
            async with _pool.connection() as conn:
                await insert_chunks_batch(conn, rows)
                await conn.commit()
            return
        except Exception as e:
            last_exc = e
            logger.warning('DB write failed (attempt %d/%d): %s', attempt + 1, max_attempts, e)
            if attempt < max_attempts - 1:
                await init_pool()
    raise last_exc

print('DB helpers ready')

In [ ]:
# ── Main pipeline ─────────────────────────────────────────────────────────────
async def run_pipeline(limit: int = LIMIT, batch_size: int = BATCH_SIZE):
    await init_pool()

    total_docs = 0
    total_chunks = 0
    start = time.time()
    buffer: list[tuple[int, str, str, list]] = []  # (paper_id, title, abstract, chunks)

    async def flush(buf):
        if not buf:
            return 0

        paper_ids_flat, chunks_flat = [], []
        for pid, _title, _abstract, chunks in buf:
            for c in chunks:
                paper_ids_flat.append(pid)
                chunks_flat.append(c)

        pairs = embed_chunks(chunks_flat)

        rows = [
            {
                'paper_id': paper_ids_flat[i],
                'section_title': chunk.section_title,
                'chunk_index': chunk.chunk_index,
                'content': chunk.content,
                'token_count': chunk.token_count,
                'embedding': emb,
            }
            for i, (chunk, emb) in enumerate(pairs)
        ]

        await write_batch_with_retry(rows)

        return len(rows)

    async with _pool.connection() as conn:
        async with conn.cursor() as cur:
            await cur.execute(
                """
                SELECT id, arxiv_id, title, abstract
                FROM papers
                WHERE abstract IS NOT NULL AND abstract != ''
                ORDER BY published_at DESC
                LIMIT %s
                """,
                (limit,),
            )
            rows_fetched = await cur.fetchall()

    logger.info('Fetched %d papers from DB', len(rows_fetched))

    for paper_id, arxiv_id, title, abstract in rows_fetched:
        chunks = chunk_text(abstract, doc_id=arxiv_id)
        if not chunks:
            continue

        buffer.append((paper_id, title, abstract, chunks))
        total_docs += 1

        if len(buffer) >= batch_size:
            n = await flush(buffer)
            total_chunks += n
            buffer = []
            elapsed = time.time() - start
            rate = total_docs / elapsed * 60
            logger.info(
                'Processed %d docs | %d chunks | %.0f docs/min | %.1fs elapsed',
                total_docs, total_chunks, rate, elapsed,
            )

    # Flush remainder
    n = await flush(buffer)
    total_chunks += n

    await close_pool()

    elapsed = time.time() - start
    stats = {
        'total_docs': total_docs,
        'total_chunks': total_chunks,
        'elapsed_minutes': round(elapsed / 60, 1),
    }
    logger.info('Pipeline complete: %s', stats)
    return stats

print('Pipeline function defined — ready to run')

In [ ]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# No LLM calls in this pipeline, so there's no free-tier rate limit to budget
# around — the only constraint is GPU embedding throughput and Neon's 512 MB
# free-tier storage limit (roughly ~10k papers of abstract-only chunks).
stats = await run_pipeline(limit=10_000, batch_size=1000)
print('\n✓ Done:', stats)